# TalentLens
Extracción de información de CVs con PLN para agilizar la preselección de candidatos.

## 1. Captura de información
Leemos el texto de los CVs en PDF y Word. Los PDF escaneados se leen con OCR.

In [7]:
from pathlib import Path

import pymupdf
import docx
import pytesseract
import pandas as pd
from PIL import Image

DATA = Path('CVs_ficticios_prueba')
archivos = sorted(list(DATA.glob('pdf/*.pdf')) + list(DATA.glob('word/*.docx')))

print(len(archivos), 'CVs encontrados')
print('Tesseract', pytesseract.get_tesseract_version())
print('Idiomas OCR:', pytesseract.get_languages())

20 CVs encontrados
Tesseract 5.5.3
Idiomas OCR: ['afr', 'amh', 'ara', 'asm', 'aze', 'aze_cyrl', 'bel', 'ben', 'bod', 'bos', 'bre', 'bul', 'cat', 'ceb', 'ces', 'chi_sim', 'chi_sim_vert', 'chi_tra', 'chi_tra_vert', 'chr', 'cos', 'cym', 'dan', 'deu', 'div', 'dzo', 'ell', 'eng', 'enm', 'epo', 'equ', 'est', 'eus', 'fao', 'fas', 'fil', 'fin', 'fra', 'frk', 'frm', 'fry', 'gla', 'gle', 'glg', 'grc', 'guj', 'hat', 'heb', 'hin', 'hrv', 'hun', 'hye', 'iku', 'ind', 'isl', 'ita', 'ita_old', 'jav', 'jpn', 'jpn_vert', 'kan', 'kat', 'kat_old', 'kaz', 'khm', 'kir', 'kmr', 'kor', 'kor_vert', 'lao', 'lat', 'lav', 'lit', 'ltz', 'mal', 'mar', 'mkd', 'mlt', 'mon', 'mri', 'msa', 'mya', 'nep', 'nld', 'nor', 'oci', 'ori', 'osd', 'pan', 'pol', 'por', 'pus', 'que', 'ron', 'rus', 'san', 'sin', 'slk', 'slv', 'snd', 'spa', 'spa_old', 'sqi', 'srp', 'srp_latn', 'sun', 'swa', 'swe', 'syr', 'tam', 'tat', 'tel', 'tgk', 'tha', 'tir', 'ton', 'tur', 'uig', 'ukr', 'urd', 'uzb', 'uzb_cyrl', 'vie', 'yid', 'yor']


### Funciones de lectura
- **PDF:** texto con pymupdf; si la página no tiene texto (escaneada) se usa OCR en español (`--psm 4` mantiene juntas las filas de las tablas).
- **Word:** párrafos y tablas en el orden del documento.

In [12]:
def leer_pdf(ruta):
    """Texto de un PDF; si una página no tiene texto, se lee con OCR."""
    paginas = []
    with pymupdf.open(ruta) as pdf:
        for pagina in pdf:
            texto = pagina.get_text()
            if len(texto.strip()) < 20:
                pix = pagina.get_pixmap(dpi=300)
                img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
                texto = pytesseract.image_to_string(img, lang='spa', config='--psm 4')
            paginas.append(texto)
    return '\n'.join(paginas)

def leer_docx(ruta):
    """Texto de un Word en orden, incluyendo tablas."""
    doc = docx.Document(ruta)
    lineas = []
    for bloque in doc.iter_inner_content():
        if isinstance(bloque, docx.table.Table):
            for fila in bloque.rows:
                celdas = []
                for celda in fila.cells:
                    if celda.text not in celdas:  # celdas combinadas se repiten
                        celdas.append(celda.text)
                lineas.append('\n'.join(celdas))
        else:
            lineas.append(bloque.text)
    return '\n'.join(lineas)

def leer_cv(ruta):
    ruta = Path(ruta)
    if ruta.suffix.lower() == '.pdf':
        return leer_pdf(ruta)
    if ruta.suffix.lower() == '.docx':
        return leer_docx(ruta)
    raise ValueError(f'Formato no soportado: {ruta.suffix}')

### Leer todos los CVs

In [9]:
cvs = pd.DataFrame({
    'archivo': [str(a.relative_to(DATA)) for a in archivos],
    'texto': [leer_cv(a) for a in archivos],
})
cvs['caracteres'] = cvs['texto'].str.len()
cvs[['archivo', 'caracteres']]

,archivo,caracteres
0,pdf/cv_01_clasico.pdf,698
1,pdf/cv_02_moderno_foto.pdf,1052
2,pdf/cv_03_tabla.pdf,1060
3,pdf/cv_05_creativo_foto.pdf,840
4,pdf/cv_06_clasico.pdf,1205
5,pdf/cv_07_escaneado.pdf,888
6,pdf/cv_09_moderno_foto.pdf,753
7,pdf/cv_10_varias_paginas.pdf,2206
8,pdf/cv_11_creativo_foto.pdf,685
9,pdf/cv_13_tabla.pdf,1101
